In [31]:
import os
import pandas as pd
import numpy as np

In [50]:
mapping = {0.0:0,0.2:20,
        0.4:40,0.6:60,
        0.8:80,1.0:100}
def get_value(val):
    return mapping.get(val, val)

mapping2 = {'MNIST':'MNIST','FashionMNIST':'Fashion-MNIST',
    'CIFAR10':'CIFAR-10','CIFAR100':'CIFAR-100',
    'TinyImageNet':'Tiny ImageNet'}

def get_dataname(val):
    return mapping2.get(val, val)

def fun_1(train_overall,classnumber=99):
    if classnumber==99:
        max_value = list(map(max, zip(train_overall[0], train_overall[1],train_overall[2])))
        min_value = list(map(min, zip(train_overall[0], train_overall[1],train_overall[2])))
        avg_value = list(map(sum, zip(train_overall[0], train_overall[1],train_overall[2])))
        avg_value = [x/3 for x in avg_value]
    else:
        max_value = list(map(max, zip(train_overall[0][classnumber], train_overall[1][classnumber],train_overall[2][classnumber])))
        min_value = list(map(min, zip(train_overall[0][classnumber], train_overall[1][classnumber],train_overall[2][classnumber])))
        avg_value = list(map(sum, zip(train_overall[0][classnumber], train_overall[1][classnumber],train_overall[2][classnumber])))
        avg_value = [x/3 for x in avg_value]
    return max_value,min_value,avg_value


def path(network,data_type,corrupt,n):    
    if n==0.99:
        typex=f'../pca_during_training_angle/angle_results/{data_type}_{network}/results_{corrupt}'
        type_acc=f'../pca_during_training_angle/angle_results/{data_type}_{network}/Accuracy_results/{corrupt}'
    if n==1:
        typex=f'../pca_during_training_angle/angle_results_1pc/{data_type}_{network}/results_{corrupt}'
        type_acc=f'../pca_during_training_angle/angle_results/{data_type}_{network}/Accuracy_results/{corrupt}'
    if network=='ResNet18':
        typex=f'../TMLR_ResNet18_during_training/results/40/angle_results_{n}/{data_type}_{network}/results_{corrupt}'
        type_acc=f'../TMLR_compare/Modern_backbones/models/{data_type}_{network}/Accuracy_results/{corrupt}'
    return typex,type_acc

def divide_function(list_value,int_value):
    newList = []
    for x in list_value:
        newList.append(x/int_value)
        
    return newList
def xaxis_ticks_print(list_value):
    new_xvalues=list_value[::10]
    new_xvalues.append(list_value[-1])
    return new_xvalues

def accuracy_model(type_acc,data_name,path_x_values,corrupt,tiny=False):
    if data_name=='Corrupted Training':
        data='Train_accuracy'
    if data_name=='Original Training':
        data='Train_accuracy_org'
    if data_name=='Testing':
        data='Test_accuracy'
    

    len_x_run=[]
    last_values_xaxis=[]
    run_list=[]
    for run in range(1,4,1):
        
        if data_name=='Corrupted Training' or data_name=='Testing':
            run_type=f'Run_{run}'
        elif data_name=='Original Training' and corrupt==0.0:
            run_type=f'Run_{run}'
            data='Train_accuracy'
        elif data_name=='Original Training' and corrupt!=0.0:
            run_type=f'Run_{run}_org'
            
        filename_read=f'{type_acc}/{run_type}.csv'
        num_epoch=(pd.read_csv(filename_read)
                   .sort_values(by='epoch')['epoch']).tolist()[-1]
        x_acc=[i for i in range(-1,num_epoch,1)]
        x_acc[0]=0

        run_list.append(divide_function(pd.read_csv(filename_read)
                                   .sort_values(by='epoch')[data]
                                   [select_model(x_acc,tiny)].tolist(),100))
        path_model_num=f'{path_x_values}/angle_results/Run_{run}/'
        len_x_run.append(len(os.listdir(path_model_num)))
        if tiny:
            last_values_xaxis.append(sorted(os.listdir(path_model_num),key=int)[-1])


    len_x=min(len_x_run)
    ind_len=len_x_run.index(len_x)                     
    path_model=f'{path_x_values}/angle_results/Run_{ind_len+1}/'      
    if tiny:
        values_xaxis=sorted(os.listdir(path_model),key=int)[:-1]
        values_xaxis=values_xaxis+last_values_xaxis
    else:
        values_xaxis=sorted(os.listdir(path_model),key=int)

    x=[i for i in range(len_x)]
    return run_list,values_xaxis,x



def select_model(values,tiny=False):
    selected_values = []
    # Starting index and increment
    i = 0
    increment = 2

    # Loop through the list with dynamic increments
    while i < len(values):
        if i==0:
            selected_values.append(values[i])
            selected_values.append(values[i+1])
        else:
            selected_values.append(values[i])
        i += increment
        # Update increment every time it reaches 20 to increase by 5
        if i < 20:
            increment = 2
        else:
            if tiny==True:
                increment= 10
            else:
                increment = 5
            
    return selected_values

def accuracy_model_tiny(type_acc,data_name,path_x_values,corrupt,tiny=False):

    run=[]
    if data_name=='Corrupted Training':
        data='Train_accuracy'
    if data_name=='Testing':
        data='Test_accuracy'


    len_x_run=[]
    last_values_xaxis=[]

    for run_type in range(1,4):  
        num_epoch,min_epoch=final_epoch_tiny(corrupt,run_type)
        x_acc=[i for i in range(-1,min_epoch,1)]+[num_epoch]
        x_acc[0]=0
        if corrupt==0.8 or corrupt==1.0:
            models_load=select_model(x_acc,tiny)
        else:
            models_load=select_model(x_acc,tiny)+[x_acc[-1]-1]        
        run.append(divide_function(pd.read_csv(f'{type_acc}/Run_{run_type}.csv')
                                   .sort_values(by='epoch')[data]
                                   [models_load].tolist(),100))

        path_model_num=f'{path_x_values}/angle_results/Run_{run_type}/'
        len_x_run.append(len(os.listdir(path_model_num)))
        if tiny:
            last_values_xaxis.append(sorted(os.listdir(path_model_num),key=int)[-1])
    len_x=min(len_x_run)
    x=[i for i in range(len_x)]
    ind_len=len_x_run.index(len_x)                     
    path_model=f'{path_x_values}/angle_results/Run_{ind_len+1}/'      
    if tiny:
        values_xaxis=sorted(os.listdir(path_model),key=int)[:-1]
        values_xaxis=values_xaxis+last_values_xaxis
    else:
        values_xaxis=sorted(os.listdir(path_model),key=int)

    return run,values_xaxis,x

def final_epoch_tiny(corrupt,run):
    if corrupt==0.0:
        epoch=[175,178,167]
    if corrupt==0.2:
        epoch=[184,194,193]
    if corrupt==0.4:
        epoch=[263,240,272]
    if corrupt==0.6:
        epoch=[449,426,428]
    if corrupt==0.8 or corrupt==1.0:
        epoch=[499,499,499]

    return epoch[run-1]+1,min(epoch)+1



def read_file_name(data_name,type_cnn,folder_name,run,model_number,layer):
    if data_name == 'Testing' :
        file_name=f'{type_cnn}/{folder_name}/Run_{run}/{model_number}/layer_{layer}_acc_overall_test.csv'
    if data_name == 'Corrupted Training' :
        file_name=f'{type_cnn}/{folder_name}/Run_{run}/{model_number}/layer_{layer}_acc_overall_train.csv'
    if data_name == 'Original Training' :
        file_name=f'{type_cnn}/angle_results_exp2/Run_{run}/{model_number}/layer_{layer}_acc_overall_train.csv'
    return file_name
def accuracy_model_all(type_acc,data_name,path_x_values,corrupt,tiny=False):
    run=[]
    if data_name=='Corrupted Training':
        data='Train_accuracy'

    if data_name=='Testing':
        data='Test_accuracy'
        
    len_x_run=[]
    last_values_xaxis=[]
    for run_type in ['Run_1','Run_2','Run_3']:
        
        num_epoch=(pd.read_csv(f'{type_acc}/{run_type}.csv')
                   .sort_values(by='epoch')['epoch']).tolist()[-1]
        x_acc=[i for i in range(-1,num_epoch,1)]
        x_acc[0]=0
        
        if data_name=='Original Training':
            if corrupt ==0.0:
                data='Train_accuracy'
                run.append(divide_function(pd.read_csv(f'{type_acc}/{run_type}.csv')
                                           .sort_values(by='epoch')[data]
                                           [select_model(x_acc,tiny)].tolist(),100))
            else:
                data='Train_accuracy_org'
                run.append(divide_function(pd.read_csv(f'{type_acc}/{run_type}_org.csv')
                                           .sort_values(by='epoch')[data]
                                           [select_model(x_acc,tiny)].tolist(),100))
        
        else:
            run.append(divide_function(pd.read_csv(f'{type_acc}/{run_type}.csv')
                                           .sort_values(by='epoch')[data]
                                           [select_model(x_acc,tiny)].tolist(),100))
        path_model_num=f'{path_x_values}/angle_results/{run_type}/'
        len_x_run.append(len(os.listdir(path_model_num)))
        if tiny:
            last_values_xaxis.append(sorted(os.listdir(path_model_num),key=int)[-1])
    
    
    len_x=min(len_x_run)
    ind_len=len_x_run.index(len_x)                     
    path_model=f'{path_x_values}/angle_results/Run_{ind_len+1}/'      
    if tiny:
        values_xaxis=sorted(os.listdir(path_model),key=int)[:-1]
        values_xaxis=values_xaxis+last_values_xaxis
    else:
        values_xaxis=sorted(os.listdir(path_model),key=int)
    
    x=[i for i in range(len_x)]
    return run,values_xaxis,x

def results_name(file_name):
    os.makedirs(f'{file_name}',exist_ok=True)
    os.makedirs(f'{file_name}/ACCURACY',exist_ok=True)
    os.makedirs(f'{file_name}/ACCURACY/pdf',exist_ok=True)
    os.makedirs(f'{file_name}/ACCURACY/png',exist_ok=True)

    result1=f'{file_name}/ACCURACY/pdf'
    result2=f'{file_name}/ACCURACY/png'
    
    return result1,result2
def ntype_folder_name(subspace_type,n):
    if subspace_type=='corrupt':
        folder_name='angle_results'
    if subspace_type=='original':
        folder_name='angle_results_exp3'
        
    if n==0.99:
        n_type='angle_results'
    if n==1:
        n_type='angle_results_1pc'
    return folder_name,n_type

def main_appendix(subspace_type,data_name,n):
    networks=['MLP','MLP','CNN','CNN','CNN']
    data_types=['MNIST','CIFAR10','MNIST','FashionMNIST','CIFAR10']

    labels_names_total=[f'Model test accuracy','Input/ ResNet-18-L1',
                    'MLP-FC1 (128)/ CNN-Flat (576/1024)/ AlexNet-Flat (256)/ ResNet-18-L2',
                    'MLP-FC2 (512)/ CNN-FC1 (250)/ AlexNet-FC1 (4096)/ ResNet-18-L3',
                    'MLP-FC3 (2048)/ CNN-FC2 (250)/ AlexNet-FC2 (4096)/ ResNet-18-L4',
                    'MLP-FC4 (2048)/ CNN-FC3 (250)/ ResNet-18-Avg_pool']  
    file_final_name=f'ACCURACY_{subspace_type}_{data_name}_{n}_all'
    
    return networks,data_types,labels_names_total,file_final_name

def layer_info(network):
    if network =='MLP':
        labels_names=[f'Model test accuracy','Input',
                      'FC1 (128)','FC2 (512)','FC3 (2048)','FC4 (2048)']        
        data_layer_name=['fc1','fc2','fc3','fc4']     #'input',           

    elif network =='CNN': 
        labels_names=[f'Model test accuracy','Input',
                  'Flat (576/1024)',
                  'FC1 (250)','FC2 (250)','FC3 (250)']        

        data_layer_name=['flattern','fc1','fc2','fc3']#'input',
    elif network =='AlexNet':     
        labels_names=[f'Model test accuracy','Flat (256)',
                      'FC1 (4096)','FC2 (4096)']        
        data_layer_name=['flattern','fc1','fc2']
        
    elif network =='ResNet18':       
        labels_names = ['Model test accuracy','L1',
                        'L2','L3','L4','Avg_pool'] #'L0'
        data_layer_name = ['l1','l2','l3','l4','bf_last']#'lo'
    return labels_names,data_layer_name

def print_x_axis(values_xaxis):
    x_labels = list(map(int, values_xaxis))  
    max_val = max(x_labels)

    # Decide step size adaptively
    if max_val <= 40:
        tick_labels = list(range(0, max_val + 1, 10))
    else:
        tick_labels = list(range(0, max_val + 1, 40))

    # Find closest indices in x_labels for each desired tick
    tick_indices = [min(range(len(x_labels)), key=lambda i: abs(x_labels[i] - val))
                    for val in tick_labels if val <= max_val]

    return tick_labels, tick_indices, x_labels


In [64]:
# Model test accuracy


    
label_text='% corruption degree'
subspace_type='corrupt'
n=0.99
folder_name,n_type=ntype_folder_name(subspace_type,n)
data_name='Testing'
corruption=[0.0,0.2,0.4,0.6,0.8,1.0]
y_label=f'Minimum Angle Subspace Classifier (MASC) Test Accuracy'

print(f'Model-dataset | Model accuracy ')

for epoch in [0,-1]:
    
    networks,data_types,labels_names_total,file_final_name=main_appendix(subspace_type,data_name,n)
    for network,data_type in zip(networks,data_types):
        print(f'{network}-{get_dataname(data_type)} ')
        for corrupt in corruption:
            labels_names,data_layer_name=layer_info(network)   

            type_cnn,type_acc=path(network,data_type,corrupt,n)

            run_acc,values_xaxis,x=accuracy_model(type_acc,data_name,
                                                  type_cnn,corrupt)
            _,_,avg_value=fun_1(run_acc)

            avg_value[0]=0.10

            model_test=avg_value[epoch]
            print(corrupt,round(model_test*100,2))

    network='AlexNet'
    data_type='TinyImageNet'
    print(f'{network}-{get_dataname(data_type)}')

    for corrupt in corruption:
        labels_names,data_layer_name=layer_info(network)
        type_cnn,type_acc=path(network,data_type,corrupt,n)

        run_acc,values_xaxis,x=accuracy_model_tiny(type_acc,data_name,
                                              type_cnn,corrupt,tiny=True)

        _,_,avg_value=fun_1(run_acc)

        avg_value[0]=0.005
        model_test=avg_value[epoch]
        print(corrupt,round(model_test*100,2))

    

    network='ResNet18'
    data_type='CIFAR10'
    print(f'ResNet-18-{get_dataname(data_type)} ')
    for corrupt in corruption:       
        if corrupt==1.0:
            continue
        labels_names,data_layer_name=layer_info(network)   

        type_cnn,type_acc=path(network,data_type,corrupt,n)
        run_acc,values_xaxis,x=accuracy_model(type_acc,data_name,
                                              type_cnn,corrupt)

        max_value,min_value,avg_value=fun_1(run_acc)

        avg_value[0]=0.10
        model_test=avg_value[epoch]
        print(corrupt,round(model_test*100,2))

    print("\n")





Model-dataset | Model accuracy 
MLP-MNIST |
0.0 10.0 |
0.2 10.0 |
0.4 10.0 |
0.6 10.0 |
0.8 10.0 |
1.0 10.0 |
MLP-CIFAR-10 |
0.0 10.0 |
0.2 10.0 |
0.4 10.0 |
0.6 10.0 |
0.8 10.0 |
1.0 10.0 |
CNN-MNIST |
0.0 10.0 |
0.2 10.0 |
0.4 10.0 |
0.6 10.0 |
0.8 10.0 |
1.0 10.0 |
CNN-Fashion-MNIST |
0.0 10.0 |
0.2 10.0 |
0.4 10.0 |
0.6 10.0 |
0.8 10.0 |
1.0 10.0 |
CNN-CIFAR-10 |
0.0 10.0 |
0.2 10.0 |
0.4 10.0 |
0.6 10.0 |
0.8 10.0 |
1.0 10.0 |
AlexNet-Tiny ImageNet |
0.0 0.5 |
0.2 0.5 |
0.4 0.5 |
0.6 0.5 |
0.8 0.5 |
1.0 0.5 |
ResNet-18-CIFAR-10 |
0.0 10.0 |
0.2 10.0 |
0.4 10.0 |
0.6 10.0 |
0.8 10.0 |


MLP-MNIST |
0.0 97.88 |
0.2 87.73 |
0.4 72.76 |
0.6 52.89 |
0.8 31.16 |
1.0 10.21 |
MLP-CIFAR-10 |
0.0 53.32 |
0.2 44.17 |
0.4 35.85 |
0.6 26.95 |
0.8 17.7 |
1.0 9.9 |
CNN-MNIST |
0.0 99.08 |
0.2 88.34 |
0.4 68.59 |
0.6 47.7 |
0.8 28.52 |
1.0 10.45 |
CNN-Fashion-MNIST |
0.0 91.01 |
0.2 79.07 |
0.4 60.14 |
0.6 42.21 |
0.8 25.61 |
1.0 10.16 |
CNN-CIFAR-10 |
0.0 74.96 |
0.2 60.48 |
0.4 46.15 |
0.6 30.9

In [65]:

label_text='% corruption degree'
subspace_type='corrupt'
n=0.99
folder_name,n_type=ntype_folder_name(subspace_type,n)
data_name='Testing'
corruption=[0.0,0.2,0.4,0.6,0.8,1.0]
y_label=f'Minimum Angle Subspace Classifier (MASC) Test Accuracy'

print(f'Model-dataset | {get_value(corruption[0])}{label_text} | {get_value(corruption[1])}{label_text} | {get_value(corruption[2])}{label_text} | {get_value(corruption[3])}{label_text} | {get_value(corruption[4])}{label_text} |  {get_value(corruption[5])}{label_text}')
for epoch in [0,-1]:
    networks,data_types,labels_names_total,file_final_name=main_appendix(subspace_type,data_name,n)
    for network,data_type in zip(networks,data_types):
        print(f'{network}-{get_dataname(data_type)}')
        for corrupt in corruption:
            labels_names,data_layer_name=layer_info(network)   

            type_cnn,type_acc=path(network,data_type,corrupt,n)

            run_acc,values_xaxis,x=accuracy_model(type_acc,data_name,
                                                  type_cnn,corrupt)

            test_accuracies=[]
            test_acc_layer=[]
            for layer,labels_name in zip(data_layer_name,labels_names[1:]):

                run_layer=[]
                for run in range(1,4):
                    run_value=[]
                    for model_number in values_xaxis:
                        file_name_final=read_file_name(data_name,type_cnn,
                                                       folder_name,run,
                                                       model_number,layer)
                        run_value.append(pd.read_csv(file_name_final)['acc_overall'][0])
                    run_layer.append(run_value)
                _,_,avg_value=fun_1(run_layer)
                test_accuracies.append(avg_value[epoch])
                test_acc_layer.append(layer)
            argmax_index = np.argmax(np.array(test_accuracies))
            print(corrupt,round(test_accuracies[argmax_index]*100,2),test_acc_layer[argmax_index])

    network='AlexNet'
    data_type='TinyImageNet'
    print(f'{network}-{get_dataname(data_type)}')

    for corrupt in corruption:       

        labels_names,data_layer_name=layer_info(network)
        type_cnn,type_acc=path(network,data_type,corrupt,n)

        run_acc,values_xaxis,x=accuracy_model_tiny(type_acc,data_name,
                                              type_cnn,corrupt,tiny=True)

        for layer,labels_name in zip(data_layer_name,labels_names[1:]):
            run_layer=[]
            for run in range(1,4):
                run_value=[]
                for model_number in values_xaxis[:-3]:
                    file_name_final=read_file_name(data_name,type_cnn,
                                                           folder_name,run,
                                                           model_number,layer)
                    run_value.append(pd.read_csv(file_name_final)['acc_overall'][0])

                model_number= values_xaxis[-4+run]
                file_name_final=read_file_name(data_name,type_cnn,
                                                           folder_name,run,
                                                           model_number,layer)
                run_value.append(pd.read_csv(file_name_final)['acc_overall'][0])

                run_layer.append(run_value)
            _,_,avg_value=fun_1(run_layer)
            test_accuracies.append(avg_value[epoch])
            test_acc_layer.append(layer)
        argmax_index = np.argmax(np.array(test_accuracies))
        print(corrupt,round(test_accuracies[argmax_index]*100,2),test_acc_layer[argmax_index])



    network='ResNet18'
    data_type='CIFAR10'
    print(f'ResNet-18-{get_dataname(data_type)}')

    for corrupt in corruption:       
        if corrupt==1.0:
            continue
        labels_names,data_layer_name=layer_info(network)   

        type_cnn,type_acc=path(network,data_type,corrupt,n)
        run_acc,values_xaxis,x=accuracy_model(type_acc,data_name,
                                              type_cnn,corrupt)

        for layer,labels_name in zip(data_layer_name,labels_names[1:]):

            run_layer=[]
            for run in range(1,4):
                run_value=[]
                for model_number in values_xaxis:
                    file_name_final=read_file_name(data_name,type_cnn,
                                                   folder_name,run,
                                                   model_number,layer)
                    run_value.append(pd.read_csv(file_name_final)['acc_overall'][0])
                run_layer.append(run_value)
            _,_,avg_value=fun_1(run_layer)
            test_accuracies.append(avg_value[epoch])
            test_acc_layer.append(layer)
        argmax_index = np.argmax(np.array(test_accuracies))
        print(corrupt,round(test_accuracies[argmax_index]*100,2),test_acc_layer[argmax_index])


    print("\n")



Model-dataset | 0% corruption degree | 20% corruption degree | 40% corruption degree | 60% corruption degree | 80% corruption degree |  100% corruption degree
MLP-MNIST |
0.0 88.03 fc4 |
0.2 87.3 fc4 |
0.4 87.48 fc4 |
0.6 88.29 fc4 |
0.8 85.88 fc4 |
1.0 10.21 fc3 |
MLP-CIFAR-10 |
0.0 36.76 fc3 |
0.2 36.07 fc3 |
0.4 35.71 fc3 |
0.6 34.32 fc3 |
0.8 28.62 fc3 |
1.0 10.07 fc3 |
CNN-MNIST |
0.0 82.89 flattern |
0.2 82.33 flattern |
0.4 81.23 flattern |
0.6 79.4 flattern |
0.8 76.97 flattern |
1.0 17.8 fc2 |
CNN-Fashion-MNIST |
0.0 67.81 flattern |
0.2 67.02 flattern |
0.4 65.7 flattern |
0.6 62.82 flattern |
0.8 61.84 flattern |
1.0 13.47 flattern |
CNN-CIFAR-10 |
0.0 30.11 flattern |
0.2 30.16 flattern |
0.4 29.87 flattern |
0.6 29.31 flattern |
0.8 27.45 flattern |
1.0 10.23 fc1 |
AlexNet-Tiny ImageNet |
0.0 10.23 fc1 |
0.2 10.23 fc1 |
0.4 10.23 fc1 |
0.6 10.23 fc1 |
0.8 10.23 fc1 |
1.0 10.23 fc1 |
ResNet-18-CIFAR-10 |
0.0 62.7 l1 |
0.2 62.7 l1 |
0.4 62.7 l1 |
0.6 62.7 l1 |
0.8 62.7 l1 |
